# ModernBERT - Feature attribution analysis
## Integrated Gradients, GradientSHAP, and Attention Rollout on full validation sets

ModernBERT counterpart of `attribution.ipynb`.
The best-seed ModernBERT models from `baseline_modernbert.ipynb` are loaded and
the same three attribution methods are run on the full validation sets of both datasets.

### Attribution methods
1. **Integrated Gradients (IG)** - gradient-based, Captum
2. **GradientSHAP (SHAP)** - gradient-based, Captum
3. **Attention Rollout (AR)** - computed via `output_attentions=True` with `attn_implementation="eager"`

### ModernBERT-specific adaptations
| Step | BERT | ModernBERT |
|---|---|---|
| Embedding layer | `model.bert.embeddings` | `model.model.embeddings` |
| Attention rollout | Manual Q/K from `sa.query` / `sa.key` | `output_attentions=True` (eager impl) |
| `token_type_ids` | Present | Not used - omitted |
| Max token length | 512 | No cap (model native 8192) |

### Prerequisites
- `baseline_modernbert.ipynb` must have run successfully
- Required files in `runs/`:
  - `results/pan25_multiclass_modernbert_seeds.csv`
  - `results/authormix_modernbert_seeds.csv`
  - `models/pan25_multiclass_modernbert_best_seed*/`
  - `models/authormix_modernbert_best_seed*/`
  - `models/pan25_multiclass_modernbert_label_map.json`
  - `cache/pan25_multiclass_tok_modernbert/`
  - `cache/authormix_tok_modernbert/`

## 1. Setup

In [ ]:
import glob, shutil
for p in glob.glob("/usr/local/lib/python*/dist-packages/~*"):
    shutil.rmtree(p, ignore_errors=True)
    print(f"Removed corrupted partial install: {p}")

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "transformers>=4.48.0",
                "captum", "datasets"],
               check=True)

import os
os.kill(os.getpid(), 9)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
# reduce GPU memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

Mounted at /content/drive


In [ ]:
import os, json, math, re, time, gc
import numpy as np
import pandas as pd
import torch

def free_gpu():
    """Release GPU memory caches between heavy inference calls."""
    gc.collect()
    torch.cuda.empty_cache()

from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import IntegratedGradients, GradientShap

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words("english"))

ROOT      = "/content/drive/MyDrive/ap-thesis"
RUNS_ROOT = f"{ROOT}/runs"
os.makedirs(f"{RUNS_ROOT}/results", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("numpy:",  np.__version__)
print("pandas:", pd.__version__)

DEVICE: cuda
GPU: Tesla T4
numpy: 2.0.2
pandas: 2.2.2


Models are loaded with `attn_implementation="eager"` so that `output_attentions=True`
returns attention weight tensors for Attention Rollout.  
Eager attention is slower than Flash/SDPA but required for this feature.

In [ ]:
def load_best_seed(csv_path, f1_col="eval_f1_macro"):
    df = pd.read_csv(csv_path)
    assert f1_col in df.columns, f"Missing column '{f1_col}' in {csv_path}"
    return int(df.sort_values(f1_col, ascending=False).iloc[0]["seed"])


# PAN25 multiclass
best_seed_pmc = load_best_seed(f"{RUNS_ROOT}/results/pan25_multiclass_modernbert_seeds.csv")
best_dir_pmc  = f"{RUNS_ROOT}/models/pan25_multiclass_modernbert_best_seed{best_seed_pmc}"
tok_dir_pmc   = f"{RUNS_ROOT}/cache/pan25_multiclass_tok_modernbert"

with open(f"{RUNS_ROOT}/models/pan25_multiclass_modernbert_label_map.json") as f:
    label_map_pmc = {int(k): v for k, v in json.load(f).items()}

pan_tok   = AutoTokenizer.from_pretrained(best_dir_pmc, use_fast=True)
pan_model = AutoModelForSequenceClassification.from_pretrained(
    best_dir_pmc,
    attn_implementation="eager",  # needed for output_attentions in AR
    # No torch_dtype: models must stay fp32 for Captum gradient attribution.
    # fp16 causes CUDA device-side asserts during IG/GradientSHAP backprop
    # because attention logits overflow fp16 range on long sequences.
    # compute_per_class_accuracy handles memory via autocast + batch_size=1.
).to(DEVICE)
pan_val   = load_from_disk(tok_dir_pmc)["validation"]

print(f"PAN25 best model: {best_dir_pmc}")
print(f"PAN25 validation size: {len(pan_val)}")


# AuthorMix
best_seed_am = load_best_seed(f"{RUNS_ROOT}/results/authormix_modernbert_seeds.csv")
best_dir_am  = f"{RUNS_ROOT}/models/authormix_modernbert_best_seed{best_seed_am}"
tok_dir_am   = f"{RUNS_ROOT}/cache/authormix_tok_modernbert"

with open(f"{best_dir_am}/label_map.json") as f:
    label_map_am = {int(k): v for k, v in json.load(f).items()}

am_tok   = AutoTokenizer.from_pretrained(best_dir_am, use_fast=True)
am_model = AutoModelForSequenceClassification.from_pretrained(
    best_dir_am,
    attn_implementation="eager",
).to(DEVICE)
am_val   = load_from_disk(tok_dir_am)["validation"]

print(f"AuthorMix best model: {best_dir_am}")
print(f"AuthorMix validation size: {len(am_val)}")

if DEVICE == "cuda":
    alloc = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU after loading both models: {alloc:.2f} GB / {total:.2f} GB total")
    print("(Expected ~1.2 GB for two fp32 149M-param models - verify runtime is fresh)")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

PAN25 best model: /content/drive/MyDrive/ap-thesis/runs/models/pan25_multiclass_modernbert_best_seed2022
PAN25 validation size: 3556


Loading weights:   0%|          | 0/138 [00:01<?, ?it/s]

AuthorMix best model: /content/drive/MyDrive/ap-thesis/runs/models/authormix_modernbert_best_seed11
AuthorMix validation size: 3642

GPU after loading both models: 1.20 GB / 15.64 GB total
(Expected ~1.2 GB for two fp32 149M-param models - verify runtime is fresh)


## 2. Balanced accuracy (mean +/- std across seeds) and per-class accuracy

In [ ]:
pan_seeds_df = pd.read_csv(f"{RUNS_ROOT}/results/pan25_multiclass_modernbert_seeds.csv")
am_seeds_df  = pd.read_csv(f"{RUNS_ROOT}/results/authormix_modernbert_seeds.csv")

pan_bal_acc = pan_seeds_df["eval_balanced_accuracy"]
am_bal_acc  = am_seeds_df["eval_balanced_accuracy"]

print(f"PAN25 multiclass (ModernBERT) balanced accuracy across {len(pan_bal_acc)} seeds:")
print(f"  Mean: {pan_bal_acc.mean():.4f}  Std: {pan_bal_acc.std():.4f}")
print(f"  Per seed: {pan_bal_acc.values}")
print()
print(f"AuthorMix (ModernBERT) balanced accuracy across {len(am_bal_acc)} seeds:")
print(f"  Mean: {am_bal_acc.mean():.4f}  Std: {am_bal_acc.std():.4f}")
print(f"  Per seed: {am_bal_acc.values}")

bal_acc_summary = pd.DataFrame([
    {"dataset": "PAN25 multiclass", "model": "ModernBERT", "n_seeds": len(pan_bal_acc),
     "mean_balanced_accuracy": round(pan_bal_acc.mean(), 4),
     "std_balanced_accuracy":  round(pan_bal_acc.std(), 4),
     "best_seed": best_seed_pmc},
    {"dataset": "AuthorMix", "model": "ModernBERT", "n_seeds": len(am_bal_acc),
     "mean_balanced_accuracy": round(am_bal_acc.mean(), 4),
     "std_balanced_accuracy":  round(am_bal_acc.std(), 4),
     "best_seed": best_seed_am},
])

out = f"{RUNS_ROOT}/results/balanced_accuracy_modernbert_summary.csv"
bal_acc_summary.to_csv(out, index=False)
print(f"\nSaved: {out}")
display(bal_acc_summary)

PAN25 multiclass (ModernBERT) balanced accuracy across 3 seeds:
  Mean: 0.7967  Std: 0.0065
  Per seed: [0.80238782 0.79803038 0.7896052 ]

AuthorMix (ModernBERT) balanced accuracy across 3 seeds:
  Mean: 0.8784  Std: 0.0041
  Per seed: [0.88183662 0.87390596 0.87931232]

Saved: /content/drive/MyDrive/ap-thesis/runs/results/balanced_accuracy_modernbert_summary.csv


,dataset,model,n_seeds,mean_balanced_accuracy,std_balanced_accuracy,best_seed
0,PAN25 multiclass,ModernBERT,3,0.7967,0.0065,2022
1,AuthorMix,ModernBERT,3,0.8784,0.0041,11


In [ ]:
def compute_per_class_accuracy(model, dataset, label_map, batch_size=1):
    """
    Batch inference on the full validation set → per-class accuracy.

    Default batch_size=1 for PAN25 (call with batch_size=32 for AuthorMix).

    Memory budget on T4 (15 GB) with fp16 models:
      Both models loaded in fp16 ≈ 300 MB each = 600 MB total.
      Eager attention QK matrix: B × 12 heads × T × T × 2 bytes.
        batch_size=1, T=3397: 1 × 12 × 3397² × 2 = 277 MB  ← safe
        batch_size=4, T=3397: 1.1 GB  ← risky given other allocations
      batch_size=1 ensures we never exceed the free budget even at max length.
    """
    model.eval()
    all_preds, all_labels = [], []

    for start in range(0, len(dataset), batch_size):
        end   = min(start + batch_size, len(dataset))
        batch = dataset[start:end]

        ids_list  = batch["input_ids"]
        mask_list = batch["attention_mask"]
        max_len   = max(len(s) for s in ids_list)

        padded_ids  = torch.zeros(len(ids_list), max_len, dtype=torch.long)
        padded_mask = torch.zeros(len(ids_list), max_len, dtype=torch.long)
        for j, (ids, mask) in enumerate(zip(ids_list, mask_list)):
            l = len(ids)
            padded_ids[j, :l]  = torch.tensor(ids,  dtype=torch.long)
            padded_mask[j, :l] = torch.tensor(mask, dtype=torch.long)

        with torch.no_grad():
            # torch.amp.autocast replaces the deprecated torch.cuda.amp.autocast
            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                logits = model(
                    input_ids=padded_ids.to(DEVICE),
                    attention_mask=padded_mask.to(DEVICE)
                ).logits
            preds = logits.argmax(dim=-1).cpu().tolist()

        all_preds.extend(preds)
        all_labels.extend(batch["label"])

        # release the padded tensors immediately after each batch
        del padded_ids, padded_mask, logits
        torch.cuda.empty_cache()

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    rows = []
    for label_id in sorted(label_map):
        mask    = all_labels == label_id
        support = int(mask.sum())
        if support == 0:
            continue
        correct = int((all_preds[mask] == label_id).sum())
        rows.append({
            "label_id":            label_id,
            "class_name":          label_map[label_id],
            "support":             support,
            "class_accuracy":      correct / support,
        })

    df = pd.DataFrame(rows).sort_values("class_accuracy", ascending=False)
    balanced_acc = df["class_accuracy"].mean()
    print(f"  Balanced accuracy (validation): {balanced_acc:.4f}")
    return df, balanced_acc


# PAN25: long sequences (avg 864, max 3397) → batch_size=1
print("PAN25 multiclass — per-class accuracy (ModernBERT best seed):")
free_gpu()
pan_pca, pan_ba = compute_per_class_accuracy(pan_model, pan_val, label_map_pmc, batch_size=1)
pan_pca.to_csv(f"{RUNS_ROOT}/results/pan25_multiclass_modernbert_per_class_accuracy_validation.csv", index=False)
display(pan_pca)

# AuthorMix: short sequences (avg 80, max 163) → batch_size=32 is fine
print("\nAuthorMix — per-class accuracy (ModernBERT best seed):")
free_gpu()
am_pca, am_ba = compute_per_class_accuracy(am_model, am_val, label_map_am, batch_size=32)
am_pca.to_csv(f"{RUNS_ROOT}/results/authormix_modernbert_per_class_accuracy_validation.csv", index=False)
display(am_pca)

PAN25 multiclass — per-class accuracy (ModernBERT best seed):
  Balanced accuracy (validation): 0.7892


,label_id,class_name,support,class_accuracy
12,12,human,1365,0.995604
9,9,gpt-4.5-preview,50,0.960000
0,0,deepseek-r1-distill-qwen-32b,132,0.946970
20,20,o3-mini,187,0.941176
6,6,gpt-3.5-turbo,212,0.915094
11,11,gpt-4o-mini,192,0.895833
16,16,llama-3.3-70b-instruct,61,0.885246
15,15,llama-3.1-8b-instruct,163,0.883436
8,8,gpt-4-turbo-paraphrase,37,0.837838
1,1,falcon3-10b-instruct,129,0.837209



AuthorMix — per-class accuracy (ModernBERT best seed):
  Balanced accuracy (validation): 0.9165


,label_id,class_name,support,class_accuracy
10,10,pp,17,1.000000
7,7,h,14,1.000000
11,11,qq,13,1.000000
3,3,blog30407,161,0.968944
0,0,blog11518,510,0.968627
13,13,woolf,488,0.934426
4,4,blog5546,160,0.931250
6,6,fitzgerald,885,0.928814
8,8,hemingway,504,0.918651
1,1,blog25872,60,0.916667


## 3. Core attribution functions

In [ ]:
def decode_tokens(tokenizer, input_ids_1d):
    return tokenizer.convert_ids_to_tokens(input_ids_1d.tolist())


def summarize_attr(attr_3d):
    """Reduce [1, seq, hidden] attribution to [seq] via abs-sum over hidden dim."""
    return attr_3d.squeeze(0).abs().sum(dim=-1).detach().cpu().numpy()


def top_tokens(tokens, scores, k=15):
    """Return top-k (token, score) pairs, excluding special tokens."""
    pairs = [(t, float(s)) for t, s in zip(tokens, scores)
             if t not in ("[CLS]", "[SEP]", "[PAD]")]
    pairs.sort(key=lambda x: abs(x[1]), reverse=True)
    return pairs[:k]


def is_punctuation(tok):
    return bool(re.fullmatch(r"\W+", tok))


def is_stopword(tok):
    return tok.lower() in STOP_WORDS


def compute_token_ratios(token_list):
    if not token_list:
        return 0.0, 0.0, 0.0
    punct   = sum(is_punctuation(t) for t in token_list)
    stop    = sum(is_stopword(t)    for t in token_list)
    content = len(token_list) - punct - stop
    n       = len(token_list)
    return content / n, punct / n, stop / n

### 3.1. Attention Rollout

**Approach**: uses `output_attentions=True` with `attn_implementation="eager"` (set at load time in section 2).

ModernBERT uses alternating local (sliding-window) and global attention layers.
With eager attention, both layer types return full `[B, H, T, T]` attention matrices - positions outside the local window are masked to near-zero after softmax, so the shape is consistent across all layers and standard rollout applies without modification.

In [ ]:
def attention_rollout_modernbert(model, ex, start_layer=0):
    """
    Attention Rollout for ModernBERT.

    Requires the model to have been loaded with attn_implementation='eager'
    so that output_attentions=True returns attention weight tensors.

    Returns CLS->token rollout scores as np.array of shape [seq_len].
    """
    model.eval()

    input_ids      = torch.tensor([ex["input_ids"]],      dtype=torch.long).to(DEVICE)
    attention_mask = torch.tensor([ex["attention_mask"]], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        out = model.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True,
            output_hidden_states=False,
        )

    if out.attentions is None:
        raise RuntimeError(
            "output_attentions returned None. "
            "Reload the model with attn_implementation='eager'."
        )

    T      = input_ids.shape[1]
    device = input_ids.device
    eye    = torch.eye(T, device=device)

    # out.attentions: tuple of (num_layers,) tensors, each [1, num_heads, T, T]
    attn_matrices = []
    for attn in out.attentions:
        probs_mean = attn.squeeze(0).mean(dim=0)  # average over heads → [T, T]
        attn_matrices.append(probs_mean)

    # Standard attention rollout: add residual, normalize, then chain-multiply
    attn_matrices = [
        (a + eye) / (a + eye).sum(dim=-1, keepdim=True)
        for a in attn_matrices
    ]

    joint = attn_matrices[start_layer]
    for i in range(start_layer + 1, len(attn_matrices)):
        joint = attn_matrices[i] @ joint

    cls_scores = joint[0].detach().cpu().numpy()  # CLS row → scores for all tokens
    return cls_scores

### 3.2. Single-example attribution (IG + GradientSHAP + AR)

The only change from `attribution_bert.ipynb` is the embedding layer path:
- BERT: `model.bert.embeddings`
- ModernBERT: `model.model.embeddings`

In [ ]:
def explain_one(model, tokenizer, ex, ig_steps=25, gs_samples=10, gs_stdevs=0.09):
    """
    Run IG + GradientSHAP + Attention Rollout on a single example.
    Input is truncated to MAX_ATTR_LEN for all three methods (see constant).
    Returns dict with tokens and top-k for each method.
    """
    model.eval()

    trunc_ids  = ex["input_ids"][:MAX_ATTR_LEN]
    trunc_mask = ex["attention_mask"][:MAX_ATTR_LEN]

    input_ids      = torch.tensor([trunc_ids],  dtype=torch.long).to(DEVICE)
    attention_mask = torch.tensor([trunc_mask], dtype=torch.long).to(DEVICE)
    target         = int(ex["label"])

    # ModernBERT: embedding layer is at model.model.embeddings (not model.bert.embeddings)
    emb_layer      = model.model.embeddings
    inputs_embeds  = emb_layer(input_ids)

    pad_id          = tokenizer.pad_token_id
    baseline_ids    = torch.full_like(input_ids, fill_value=pad_id)
    baseline_embeds = emb_layer(baseline_ids)

    def forward_embeds(embeds):
        # Expand mask to match Captum's internal batch size.
        # ModernBERT's eager_mask does not broadcast [1, T] → [B, T].
        return model(
            inputs_embeds=embeds,
            attention_mask=attention_mask.expand(embeds.shape[0], -1),
        ).logits

    # Integrated Gradients — one step at a time to avoid OOM
    ig      = IntegratedGradients(forward_embeds)
    ig_attr = ig.attribute(
        inputs=inputs_embeds, baselines=baseline_embeds,
        target=target, n_steps=ig_steps,
        internal_batch_size=1,
    )

    # GradientSHAP — manual loop (no internal_batch_size param)
    gs      = GradientShap(forward_embeds)
    gs_attr = None
    for _ in range(gs_samples):
        attr = gs.attribute(
            inputs=inputs_embeds,
            baselines=baseline_embeds,
            target=target,
            n_samples=1,
            stdevs=gs_stdevs,
        )
        gs_attr = attr.detach() if gs_attr is None else gs_attr + attr.detach()
    gs_attr = gs_attr / gs_samples

    # Attention Rollout — also truncated (out.attentions holds 22×T×T matrices)
    trunc_ex  = {"input_ids": trunc_ids, "attention_mask": trunc_mask}
    ar_scores = attention_rollout_modernbert(model, trunc_ex, start_layer=0)

    toks      = decode_tokens(tokenizer, input_ids.squeeze(0))
    ig_scores = summarize_attr(ig_attr)
    gs_scores = summarize_attr(gs_attr)

    return {
        "true_label": target,
        "tokens":     toks,
        "top_ig":     top_tokens(toks, ig_scores, k=15),
        "top_gs":     top_tokens(toks, gs_scores, k=15),
        "top_ar":     top_tokens(toks, ar_scores, k=15),
    }

## 4. Full validation attribution (3 methods)

IG, GradientSHAP, and Attention Rollout are run on every example in the validation set.
Per-example top-15 tokens and content/punct/stop ratios are saved for each method.
Output format is identical to `attribution.ipynb` for direct comparison in `attribution_enhanced.ipynb`.

In [ ]:
MAX_ATTR_LEN = 1500
"""
Hard cap for gradient-based attribution (IG, GradientSHAP) and Attention Rollout.

Why:
Backprop stores attention weights for all layers simultaneously:
  num_layers × batch × num_heads × T × T × 4 bytes (fp32)
  = 22 × 1 × 12 × T² × 4 bytes
  T = 3397 (max PAN25) → 11.8 GB — exceeds T4 (14.56 GB total).
  T = 1500             →  2.4 GB — safe with both models loaded (~1.6 GB).
Attention Rollout stores all 22 attention matrices in out.attentions:
  same formula, same cap needed.

Why 1500:
PAN25 token statistics: mean=864, p95=1256, max=3397.
At 1500 only examples above p95 are truncated (<5% of the validation set).
All three methods use the same truncated token list so their top-15 token
outputs are directly comparable.
"""


def full_validation_attribution(
    model, tokenizer, dataset, label_map,
    ig_steps=15, gs_samples=10, gs_stdevs=0.09, top_k=15,
    save_csv=None, save_json=None,
    log_every=100,
):
    """
    Run all 3 attribution methods (IG, GradientSHAP, AR) on the full dataset.

    ModernBERT adaptations vs attribution.ipynb:
    - emb_layer = model.model.embeddings
    - forward_embeds expands attention_mask to Captum's batch dim
    - IG: internal_batch_size=1  (one step at a time, avoids 14 GB OOM)
    - GradientSHAP: manual n_samples=1 loop (no internal_batch_size param)
    - All methods: inputs truncated to MAX_ATTR_LEN (see constant above)

    Returns:
        ratios_df : DataFrame with per-example content/punct/stop ratios
        all_expls : list of dicts with full token-level attributions
    """
    model.eval()
    emb_layer = model.model.embeddings
    pad_id    = tokenizer.pad_token_id

    ratio_rows = []
    all_expls  = []
    t0         = time.time()

    for i in range(len(dataset)):
        ex         = dataset[i]
        full_ids   = ex["input_ids"]
        full_mask  = ex["attention_mask"]

        # ── Truncate to MAX_ATTR_LEN for all methods ──────────────────────────
        trunc_ids  = full_ids[:MAX_ATTR_LEN]
        trunc_mask = full_mask[:MAX_ATTR_LEN]

        input_ids      = torch.tensor([trunc_ids],  dtype=torch.long).to(DEVICE)
        attention_mask = torch.tensor([trunc_mask], dtype=torch.long).to(DEVICE)
        target         = int(ex["label"])
        class_name     = label_map[target]

        inputs_embeds   = emb_layer(input_ids)
        baseline_ids    = torch.full_like(input_ids, fill_value=pad_id)
        baseline_embeds = emb_layer(baseline_ids)

        def forward_embeds(embeds):
            # Expand mask to match Captum's internal batch size.
            # ModernBERT's eager_mask does not broadcast [1, T] → [B, T].
            return model(
                inputs_embeds=embeds,
                attention_mask=attention_mask.expand(embeds.shape[0], -1),
            ).logits

        # IG
        ig      = IntegratedGradients(forward_embeds)
        ig_attr = ig.attribute(
            inputs=inputs_embeds, baselines=baseline_embeds,
            target=target, n_steps=ig_steps,
            internal_batch_size=1,
        )

        # GradientSHAP
        gs      = GradientShap(forward_embeds)
        gs_attr = None
        for _ in range(gs_samples):
            attr = gs.attribute(
                inputs=inputs_embeds,
                baselines=baseline_embeds,
                target=target,
                n_samples=1,
                stdevs=gs_stdevs,
            )
            gs_attr = attr.detach() if gs_attr is None else gs_attr + attr.detach()
        gs_attr = gs_attr / gs_samples

        # Attention Rollout
        trunc_ex = {"input_ids": trunc_ids, "attention_mask": trunc_mask}
        ar_scores = attention_rollout_modernbert(model, trunc_ex, start_layer=0)

        toks      = decode_tokens(tokenizer, input_ids.squeeze(0))
        ig_scores = summarize_attr(ig_attr)
        gs_scores = summarize_attr(gs_attr)

        top_ig = top_tokens(toks, ig_scores, k=top_k)
        top_gs = top_tokens(toks, gs_scores, k=top_k)
        top_ar = top_tokens(toks, ar_scores, k=top_k)

        ig_c, ig_p, ig_s = compute_token_ratios([t[0] for t in top_ig])
        gs_c, gs_p, gs_s = compute_token_ratios([t[0] for t in top_gs])
        ar_c, ar_p, ar_s = compute_token_ratios([t[0] for t in top_ar])

        ratio_rows.append({
            "example_idx":      i,
            "true_label":       target,
            "class":            class_name,
            "ig_content_ratio": ig_c, "ig_punct_ratio": ig_p, "ig_stop_ratio": ig_s,
            "gs_content_ratio": gs_c, "gs_punct_ratio": gs_p, "gs_stop_ratio": gs_s,
            "ar_content_ratio": ar_c, "ar_punct_ratio": ar_p, "ar_stop_ratio": ar_s,
        })

        all_expls.append({
            "example_idx": i,
            "true_label":  target,
            "class_name":  class_name,
            "tokens":      toks,
            "top_ig":      top_ig,
            "top_gs":      top_gs,
            "top_ar":      top_ar,
        })

        if (i + 1) % log_every == 0:
            elapsed = time.time() - t0
            rate    = (i + 1) / elapsed
            eta     = (len(dataset) - i - 1) / rate
            print(f"  [{i+1}/{len(dataset)}] {rate:.1f} ex/s | ETA {eta/60:.1f} min")

    ratios_df = pd.DataFrame(ratio_rows)

    if save_csv:
        ratios_df.to_csv(save_csv, index=False)
        print(f"Saved ratios CSV: {save_csv}")

    if save_json:
        with open(save_json, "w") as fh:
            json.dump(all_expls, fh)
        print(f"Saved full attributions JSON: {save_json}")

    print(f"Done: {len(dataset)} examples in {(time.time()-t0)/60:.1f} min")
    return ratios_df, all_expls

### 4.1. PAN25 full validation attribution

In [ ]:
pan_ratios, pan_expls = full_validation_attribution(
    model=pan_model,
    tokenizer=pan_tok,
    dataset=pan_val,
    label_map=label_map_pmc,
    ig_steps=15,
    gs_samples=10,
    gs_stdevs=0.09,
    save_csv=f"{RUNS_ROOT}/results/pan25_modernbert_full_validation_all_methods_ratios.csv",
    save_json=f"{RUNS_ROOT}/results/pan25_modernbert_full_validation_all_methods_attributions.json",
)

  [100/3556] 0.2 ex/s | ETA 343.8 min
  [200/3556] 0.2 ex/s | ETA 345.4 min
  [300/3556] 0.2 ex/s | ETA 333.4 min
  [400/3556] 0.2 ex/s | ETA 325.2 min
  [500/3556] 0.2 ex/s | ETA 313.3 min
  [600/3556] 0.2 ex/s | ETA 305.4 min
  [700/3556] 0.2 ex/s | ETA 295.4 min
  [800/3556] 0.2 ex/s | ETA 285.7 min
  [900/3556] 0.2 ex/s | ETA 275.8 min
  [1000/3556] 0.2 ex/s | ETA 264.8 min
  [1100/3556] 0.2 ex/s | ETA 254.9 min
  [1200/3556] 0.2 ex/s | ETA 245.6 min
  [1300/3556] 0.2 ex/s | ETA 234.5 min
  [1400/3556] 0.2 ex/s | ETA 222.8 min
  [1500/3556] 0.2 ex/s | ETA 213.0 min
  [1600/3556] 0.2 ex/s | ETA 203.3 min
  [1700/3556] 0.2 ex/s | ETA 193.1 min
  [1800/3556] 0.2 ex/s | ETA 182.5 min
  [1900/3556] 0.2 ex/s | ETA 171.9 min
  [2000/3556] 0.2 ex/s | ETA 161.2 min
  [2100/3556] 0.2 ex/s | ETA 150.9 min
  [2200/3556] 0.2 ex/s | ETA 140.7 min
  [2300/3556] 0.2 ex/s | ETA 130.5 min
  [2400/3556] 0.2 ex/s | ETA 120.3 min
  [2500/3556] 0.2 ex/s | ETA 110.1 min
  [2600/3556] 0.2 ex/s | ETA 99.8 

### 4.2. AuthorMix full validation attribution

In [ ]:
am_ratios, am_expls = full_validation_attribution(
    model=am_model,
    tokenizer=am_tok,
    dataset=am_val,
    label_map=label_map_am,
    ig_steps=15,
    gs_samples=10,
    gs_stdevs=0.09,
    save_csv=f"{RUNS_ROOT}/results/authormix_modernbert_full_validation_all_methods_ratios.csv",
    save_json=f"{RUNS_ROOT}/results/authormix_modernbert_full_validation_all_methods_attributions.json",
)

  [100/3642] 0.7 ex/s | ETA 85.1 min
  [200/3642] 0.7 ex/s | ETA 82.0 min
  [300/3642] 0.7 ex/s | ETA 79.0 min
  [400/3642] 0.7 ex/s | ETA 76.5 min
  [500/3642] 0.7 ex/s | ETA 74.0 min
  [600/3642] 0.7 ex/s | ETA 71.6 min
  [700/3642] 0.7 ex/s | ETA 69.1 min
  [800/3642] 0.7 ex/s | ETA 66.8 min
  [900/3642] 0.7 ex/s | ETA 64.5 min
  [1000/3642] 0.7 ex/s | ETA 62.0 min
  [1100/3642] 0.7 ex/s | ETA 59.6 min
  [1200/3642] 0.7 ex/s | ETA 57.1 min
  [1300/3642] 0.7 ex/s | ETA 54.5 min
  [1400/3642] 0.7 ex/s | ETA 52.1 min
  [1500/3642] 0.7 ex/s | ETA 49.6 min
  [1600/3642] 0.7 ex/s | ETA 47.3 min
  [1700/3642] 0.7 ex/s | ETA 44.9 min
  [1800/3642] 0.7 ex/s | ETA 42.5 min
  [1900/3642] 0.7 ex/s | ETA 40.2 min
  [2000/3642] 0.7 ex/s | ETA 37.8 min
  [2100/3642] 0.7 ex/s | ETA 35.5 min
  [2200/3642] 0.7 ex/s | ETA 33.2 min
  [2300/3642] 0.7 ex/s | ETA 30.8 min
  [2400/3642] 0.7 ex/s | ETA 28.5 min
  [2500/3642] 0.7 ex/s | ETA 26.2 min
  [2600/3642] 0.7 ex/s | ETA 23.9 min
  [2700/3642] 0.7 ex/

## 5. Sanity check and comparison

Ratio distributions against the BERT baseline.

In [ ]:
print(f"PAN25 ratios shape:    {pan_ratios.shape}")
print(f"AuthorMix ratios shape: {am_ratios.shape}")
print()

print("PAN25 (ModernBERT) — mean ratios by class (IG):")
display(
    pan_ratios.groupby("class")[["ig_content_ratio", "ig_punct_ratio", "ig_stop_ratio"]]
    .mean().sort_values("ig_content_ratio", ascending=False)
)
print()

print("AuthorMix (ModernBERT) — mean ratios by class (IG):")
display(
    am_ratios.groupby("class")[["ig_content_ratio", "ig_punct_ratio", "ig_stop_ratio"]]
    .mean().sort_values("ig_content_ratio", ascending=False)
)

PAN25 ratios shape:    (3556, 12)
AuthorMix ratios shape: (3642, 12)

PAN25 (ModernBERT) — mean ratios by class (IG):


,ig_content_ratio,ig_punct_ratio,ig_stop_ratio
class,,,
gemini-pro-paraphrase,0.781982,0.207207,0.010811
gemini-pro,0.769333,0.225333,0.005333
gpt-4-turbo-paraphrase,0.758559,0.219820,0.021622
mistral-7b-instruct-v0.2,0.755039,0.234109,0.010853
gpt-4-turbo,0.753623,0.231884,0.014493
gpt-3.5-turbo,0.747170,0.233648,0.019182
o3-mini,0.730125,0.260606,0.009269
gpt-4o-mini,0.724653,0.269792,0.005556
human,0.719805,0.273553,0.006642



AuthorMix (ModernBERT) — mean ratios by class (IG):


,ig_content_ratio,ig_punct_ratio,ig_stop_ratio
class,,,
blog5546,0.708155,0.271647,0.020198
bush,0.702055,0.279719,0.018225
obama,0.683220,0.296267,0.020513
blog11518,0.675485,0.302413,0.022102
blog30407,0.675166,0.289037,0.035797
fitzgerald,0.660866,0.308927,0.030207
h,0.657143,0.304762,0.038095
blog25872,0.656925,0.317024,0.026052
woolf,0.653552,0.331421,0.015027


In [ ]:
# Side-by-side comparison of IG content ratios: BERT vs ModernBERT
for tag, bert_csv, mb_ratios_df in [
    ("PAN25",
     f"{RUNS_ROOT}/results/pan25_full_validation_all_methods_ratios.csv",
     pan_ratios),
    ("AuthorMix",
     f"{RUNS_ROOT}/results/authormix_full_validation_all_methods_ratios.csv",
     am_ratios),
]:
    if not os.path.exists(bert_csv):
        print(f"BERT results not found for {tag}: {bert_csv}")
        continue

    bert_df = pd.read_csv(bert_csv)

    bert_mean = bert_df.groupby("class")["ig_content_ratio"].mean().rename("BERT")
    mb_mean   = mb_ratios_df.groupby("class")["ig_content_ratio"].mean().rename("ModernBERT")

    comparison = pd.concat([bert_mean, mb_mean], axis=1).sort_values("BERT", ascending=False)
    print(f"\n{tag} — IG content ratio by class (BERT vs ModernBERT):")
    display(comparison)

print("\nFiles saved:")
for p in [
    f"{RUNS_ROOT}/results/pan25_modernbert_full_validation_all_methods_ratios.csv",
    f"{RUNS_ROOT}/results/pan25_modernbert_full_validation_all_methods_attributions.json",
    f"{RUNS_ROOT}/results/authormix_modernbert_full_validation_all_methods_ratios.csv",
    f"{RUNS_ROOT}/results/authormix_modernbert_full_validation_all_methods_attributions.json",
]:
    print(f"  {p}")


PAN25 — IG content ratio by class (BERT vs ModernBERT):


,BERT,ModernBERT
class,,
mixtral-8x7b-instruct-v0.1,0.383333,0.718519
gpt-4.5-preview,0.380488,0.712000
gemini-pro,0.370732,0.769333
mistral-7b-instruct-v0.2,0.360000,0.755039
gemini-pro-paraphrase,0.346667,0.781982
qwen1.5-72b-chat-8bit,0.330081,0.719608
gpt-4-turbo-paraphrase,0.323810,0.758559
llama-2-70b-chat,0.318699,0.706977
llama-2-7b-chat,0.316667,0.703922



AuthorMix — IG content ratio by class (BERT vs ModernBERT):


,BERT,ModernBERT
class,,
bush,0.594519,0.702055
h,0.566667,0.657143
woolf,0.558880,0.653552
blog5546,0.551142,0.708155
qq,0.528205,0.635897
pp,0.525490,0.603922
trump,0.519931,0.652787
obama,0.497558,0.683220
hemingway,0.471958,0.651720



Files saved:
  /content/drive/MyDrive/ap-thesis/runs/results/pan25_modernbert_full_validation_all_methods_ratios.csv
  /content/drive/MyDrive/ap-thesis/runs/results/pan25_modernbert_full_validation_all_methods_attributions.json
  /content/drive/MyDrive/ap-thesis/runs/results/authormix_modernbert_full_validation_all_methods_ratios.csv
  /content/drive/MyDrive/ap-thesis/runs/results/authormix_modernbert_full_validation_all_methods_attributions.json


## 6. Agreement and ratio summary

Computes per-class mean ratios and method agreement - same format as the summary CSVs
produced in `attribution.ipynb` for direct comparison.

In [ ]:
def compute_ratio_summary(ratios_df, tag, runs_root):
    """
    Per-class mean of content/punct/stop ratios for IG, SHAP, AR.
    Saves a CSV matching the format of *_full_validation_ratio_summary.csv.
    """
    cols = [
        "ig_content_ratio", "ig_punct_ratio", "ig_stop_ratio",
        "gs_content_ratio", "gs_punct_ratio", "gs_stop_ratio",
        "ar_content_ratio", "ar_punct_ratio", "ar_stop_ratio",
    ]
    summary = ratios_df.groupby("class")[cols].mean().reset_index()
    out = f"{runs_root}/results/{tag}_modernbert_full_validation_ratio_summary.csv"
    summary.to_csv(out, index=False)
    print(f"Saved: {out}")
    return summary


def compute_agreement_summary(ratios_df, tag, runs_root):
    """
    For each example, compute which token type (content/punct/stop) dominates
    across each method, then report class-level agreement rates.
    Saves a CSV matching the format of *_full_validation_agreement_summary.csv.
    """
    methods = {
        "ig": ("ig_content_ratio", "ig_punct_ratio", "ig_stop_ratio"),
        "gs": ("gs_content_ratio", "gs_punct_ratio", "gs_stop_ratio"),
        "ar": ("ar_content_ratio", "ar_punct_ratio", "ar_stop_ratio"),
    }
    rows = []
    for m, (c_col, p_col, s_col) in methods.items():
        dom = ratios_df[[c_col, p_col, s_col]].idxmax(axis=1).str.replace(f"{m}_", "").str.replace("_ratio", "")
        tmp = ratios_df[["class"]].copy()
        tmp["dominant"] = dom
        agg = tmp.groupby(["class", "dominant"]).size().unstack(fill_value=0)
        agg = agg.div(agg.sum(axis=1), axis=0).add_prefix(f"{m}_")
        agg["method"] = m
        rows.append(agg.reset_index())
    agreement = pd.concat(rows, ignore_index=True)
    out = f"{runs_root}/results/{tag}_modernbert_full_validation_agreement_summary.csv"
    agreement.to_csv(out, index=False)
    print(f"Saved: {out}")
    return agreement


print("--- PAN25 ---")
pan_ratio_summary     = compute_ratio_summary(pan_ratios, "pan25", RUNS_ROOT)
pan_agreement_summary = compute_agreement_summary(pan_ratios, "pan25", RUNS_ROOT)

print("\n--- AuthorMix ---")
am_ratio_summary      = compute_ratio_summary(am_ratios, "authormix", RUNS_ROOT)
am_agreement_summary  = compute_agreement_summary(am_ratios, "authormix", RUNS_ROOT)

--- PAN25 ---
Saved: /content/drive/MyDrive/ap-thesis/runs/results/pan25_modernbert_full_validation_ratio_summary.csv
Saved: /content/drive/MyDrive/ap-thesis/runs/results/pan25_modernbert_full_validation_agreement_summary.csv

--- AuthorMix ---
Saved: /content/drive/MyDrive/ap-thesis/runs/results/authormix_modernbert_full_validation_ratio_summary.csv
Saved: /content/drive/MyDrive/ap-thesis/runs/results/authormix_modernbert_full_validation_agreement_summary.csv
